# 06 JEV and LAYA as judges
JEV and LAYA never hear the audio; they only reason over the evidence we send.

- **Section 1, JEV on one detector:** why FP1 adds nothing, and whether richer evidence helps. 1.1–1.3 build the
  evidence JEV could be given, 1.4 inspects what FP1 actually sent and got back, 1.5 tries richer evidence.
- **Section 2, arbitration between detectors:** when several detectors disagree, can a judge pick the right one?
  Tried with LAYA first (local, free), then JEV on the request that works best.

## Section 1: JEV on one detector's evidence

In [2]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from audio_df.data.manifests import load_manifest
from audio_df.helpers.metrics import compute_eer
from audio_df.helpers.research.arbitration_notebook import ask_jev, audio_facts, summarise
from audio_df.helpers.research.runs import collect

SR, SETS, N_RANDOM, N_DISAGREE = 16000, ["itw", "asvspoof2021_hidden", "asvspoof2021df_hidden"], 10, 10
DETECTORS = {"a1": "A1 LFCC-GMM", "a2": "A2 WavLM", "d2 L6": "D2 WavLM+LFCC one-way", "d3 L6": "D3 WavLM+LFCC two-way"}
runs = {k[0]: v["scores"] for k, v in collect("../runs", ["a1", "a2", "d2", "d3"]).items()
        if k[1] == "trimmed" and k[0] in DETECTORS}

def platt(dev):  # dev score -> P(bona fide)
    return LogisticRegression().fit(dev[["score"]], dev.label == "bonafide")

calib = {m: platt(runs[m]["dev"]) for m in DETECTORS}
thr = {m: compute_eer(runs[m]["dev"].score, runs[m]["dev"].label, higher_is_spoof=False)[1] for m in DETECTORS}

d:\MyCode\projects\GC2-EchoJEV\SystemCode\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.1 Clips
Per test set: 10 random clips plus 10 where D2 and A2 disagree at their dev thresholds, so there are mistakes to fix.
Results are reported per group; the disagreement group is deliberately hard.

In [3]:
rng = np.random.default_rng(1337)
pool = []
for s in SETS:
    merged = None
    for m in DETECTORS:
        part = runs[m][s][["clip_id", "label", "score"]].rename(columns={"score": m})
        merged = part if merged is None else merged.merge(part, on=["clip_id", "label"])
    merged["set"] = s
    disagree = (merged["d2 L6"] <= thr["d2 L6"]) != (merged["a2"] <= thr["a2"])
    pick_d = merged[disagree].sample(N_DISAGREE, random_state=1337).assign(group="disagree")
    pick_r = merged[~merged.clip_id.isin(pick_d.clip_id)].sample(N_RANDOM, random_state=1337).assign(group="random")
    pool += [pick_r, pick_d]
pool = pd.concat(pool, ignore_index=True)
for m in DETECTORS:
    pool[f"p_{m}"] = calib[m].predict_proba(pool[[m]].rename(columns={m: "score"}))[:, 1]
pool.groupby(["set", "group"]).label.value_counts().unstack(fill_value=0)

label                           bonafide  spoof
set                   group                    
asvspoof2021_hidden   disagree         5      5
                      random           0     10
asvspoof2021df_hidden disagree         1      9
                      random           0     10
itw                   disagree        10      0
                      random           9      1

## 1.2 Readable audio facts
What an analyst could read off the waveform: duration, leading silence (same rule as trimming), effective bandwidth
(highest frequency within 50 dB of the spectral peak: ~8 kHz for studio audio, ~4 kHz for telephone or narrowband)
and share of speech energy above 4 kHz (spoofs had less in notebook 03).

In [4]:
paths = {s: load_manifest(f"../data/manifests/{s}.csv", "..").set_index("utt_id").path for s in SETS}

pool = pool.join(pd.DataFrame([audio_facts(paths[r.set][r.clip_id]) for r in pool.itertuples()]))
pool.groupby("label")[["duration_s", "leading_silence_s", "bandwidth_khz", "energy_above_4khz_pct"]].median()

,duration_s,leading_silence_s,bandwidth_khz,energy_above_4khz_pct
label,,,,
bonafide,2.85,0.13,7.69,0.35
spoof,1.74,0.00,7.72,0.43


## 1.3 Rules without JEV, on the same clips
- **D2 alone:** its dev threshold.
- **WavLM average:** mean P(bona fide) of A2, D2, D3 ≥ 0.5.
- **Abstain on disagreement:** D2's verdict, or "uncertain" when D2 and A2 disagree.

In [5]:
verdict = lambda p: np.where(p >= 0.5, "bonafide", "spoof")
d2 = np.where(pool["d2 L6"] > thr["d2 L6"], "bonafide", "spoof")
a2 = np.where(pool["a2"] > thr["a2"], "bonafide", "spoof")
policies = {"D2 alone": d2,
            "WavLM average": verdict(pool[["p_a2", "p_d2 L6", "p_d3 L6"]].mean(axis=1)),
            "abstain on disagreement": np.where(d2 == a2, d2, "uncertain")}
pd.DataFrame({k: summarise(v, pool.label) for k, v in policies.items()}).T.round(3)

,coverage,accuracy on decided,errors,abstained
D2 alone,1.000,0.617,23.0,0.0
WavLM average,1.000,0.650,21.0,0.0
abstain on disagreement,0.467,0.821,5.0,32.0


## 1.4 What FP1 sent and got
FP1 (`uv run fp1`) asks JEV about one D2 score per clip: 20 clips from each of 9 test sets. Per test set: D2's
accuracy, how often JEV decided, how often JEV gave the same answer as D2, and the errors JEV corrected or
introduced.

In [2]:
FP1 = Path("../runs/E-20261028-081544-540928-fp1")
paired = pd.concat({p.stem.removeprefix("paired_"): pd.read_csv(p) for p in sorted(FP1.glob("paired_*.csv"))})
pd.DataFrame({s: {"clips": len(g), "D2 accuracy": (g.base == g.label).mean(),
                  "JEV coverage": (g.jev != "uncertain").mean(), "JEV = D2": (g.jev == g.base).mean(),
                  "corrected": int(((g.base != g.label) & (g.jev == g.label)).sum()),
                  "introduced": int(((g.base == g.label) & (g.jev != g.label) & (g.jev != "uncertain")).sum())}
              for s, g in paired.groupby(level=0)}).T.round(2)

,clips,D2 accuracy,JEV coverage,JEV = D2,corrected,introduced
asvspoof2021_eval,20.0,0.95,1.0,1.0,0.0,0.0
asvspoof2021_hidden,20.0,0.90,1.0,1.0,0.0,0.0
eval,20.0,0.90,1.0,1.0,0.0,0.0
eval_clean10k,20.0,0.90,1.0,1.0,0.0,0.0
eval_mp3_32k,20.0,0.95,1.0,1.0,0.0,0.0
eval_noise_10db,20.0,0.90,1.0,1.0,0.0,0.0
eval_opus_16k,20.0,0.90,1.0,1.0,0.0,0.0
eval_resample_8k,20.0,0.80,1.0,1.0,0.0,0.0
itw,20.0,0.50,1.0,1.0,0.0,0.0


In [3]:
request = json.loads(open(FP1 / "jev/requests.jsonl").readline())
response = json.loads(open(FP1 / "jev/responses.jsonl").readline())
print(json.dumps({"state": request["state"], "questions": list(request["questions"])}, indent=1))
print(json.dumps(response["raw_jev_response"]["answers"]["decision"], indent=1))

{
 "state": {
  "context": {
   "base_score": 0.995396077632904,
   "dev_threshold": -0.9904876351356506,
   "base_verdict": "bonafide",
   "signed_margin": 1.9858837127685547,
   "score_direction": "higher means bonafide"
  },
  "decision_guidance": "Decide whether the supplied detector evidence supports bona fide or spoof. Only one detector score is provided; do not invent acoustic evidence. Abstain when insufficient."
 },
 "questions": [
  "decision",
  "evidence_quality",
  "ambiguity"
 ]
}
{
 "type": "choice",
 "choice": "bonafide",
 "confidence": 0.95,
 "probabilities": {
  "bonafide": 0.97,
  "spoof": 0.01,
  "uncertain": 0.02
 }
}


JEV agrees with D2 on every clip: 100% coverage, nothing corrected, nothing introduced. The request explains
why. The context is one score, its threshold, and D2's own verdict (`base_verdict`), so the only consistent
answer is to repeat that verdict.

## 1.5 JEV with richer evidence
Each request carries every detector's calibrated P(bona fide), what we measured about each detector's weak spots,
and the audio facts from 1.2. It never sees the dataset name, clip ID or label. Raw D2 tokens or attention weights
are not sent: they are learned 128-number projections with no meaning a reader (or an LLM) can map to acoustics.

`RUN_JEV = False` keeps this free; set it to True to call the API (one paid call per clip not already cached in
`data/cache/jev/notebook06/`).

In [6]:
RUN_JEV = False
CACHE = Path("../data/cache/jev/notebook06")
GUIDANCE = (
    "You see evidence about one speech clip from four spoofing detectors and simple audio measurements; you do not "
    "hear the audio. Detector reliability measured on unseen web audio and newer spoofing attacks: A2 and D2 are the "
    "most reliable (~10-20% error); D3 is similar on studio audio but less reliable on web audio; A1 is unreliable "
    "on compressed or web audio (~30-58% error). bandwidth_khz near 8 means studio-quality audio; below ~4.5 means "
    "telephone or narrowband audio, where the LFCC-based A1 is least trustworthy. Leading silence is not evidence "
    "either way. Answer 'uncertain' when the reliable detectors disagree or are near 0.5.")
QUESTIONS = {
    "decision": {"type": "choice", "instructions": "Which verdict does the evidence support?",
                 "criteria": {"bonafide": "Reliable evidence favours authentic speech.",
                              "spoof": "Reliable evidence favours spoofed speech.",
                              "uncertain": "Reliable detectors conflict or are inconclusive."}},
    "evidence_quality": {"type": "score", "instructions": "How consistent is the reliable evidence?",
                         "criteria": ["Conflicting", "Mixed", "Consistent"]},
}

FACTS = ["duration_s", "leading_silence_s", "bandwidth_khz", "energy_above_4khz_pct"]

def context(r):  # evidence for one clip; no dataset name, clip id or label
    return {**{f"p_bonafide_{name.split()[0]}": round(float(r[f"p_{m}"]), 3) for m, name in DETECTORS.items()},
            **{k: float(r[k]) for k in FACTS}}

replies = [ask_jev(context(r), GUIDANCE, QUESTIONS, CACHE, RUN_JEV) for r in pool.to_dict("records")]
print(f"{sum(x is not None for x in replies)}/{len(replies)} JEV answers available")

0/60 JEV answers available


Same clips for every policy, per group: random clips and the hard disagreement clips. `D2 alone` is also
FP1's answer, since 1.4 shows JEV repeats D2.

In [7]:
if all(x is not None for x in replies):
    policies["JEV (rich evidence)"] = np.array([x["decision"]["choice"] for x in replies])
pd.concat({g: pd.DataFrame({k: summarise(v, pool.label, (pool.group == g).to_numpy()) for k, v in policies.items()}).T
           for g in ["random", "disagree"]}).round(3)

coverage  accuracy on decided  errors  \
random   D2 alone                    1.000                0.800     6.0   
         WavLM average               1.000                0.733     8.0   
         abstain on disagreement     0.933                0.821     5.0   
disagree D2 alone                    1.000                0.433    17.0   
         WavLM average               1.000                0.567    13.0   
         abstain on disagreement     0.000                0.000     0.0   

                                  abstained  
random   D2 alone                       0.0  
         WavLM average                  0.0  
         abstain on disagreement        2.0  
disagree D2 alone                       0.0  
         WavLM average                  0.0  
         abstain on disagreement       30.0

### Section 1 result
- **FP1 adds nothing.** Given one detector's score and verdict, JEV repeats the verdict on all 180 clips (1.4).
- **Richer evidence:** JEV answers were not cached in this saved run (0/60), so 1.5 has no JEV row yet.
- **The disagreements carry the signal.** On the 1.1 clips, abstaining whenever D2 and A2 disagree lifts accuracy on
  decided clips from 0.62 (D2 alone, i.e. FP1) to 0.82, at 47% coverage. Section 2 tries to resolve those
  disagreements instead of only abstaining.

## Section 2: arbitration between detectors
Six detectors judge the same In-the-Wild clip (A1 LFCC-GMM, A2 WavLM, A4 LFCC-MLP, D1, and D2/D3 with WavLM
layer 6); when they disagree, a judge decides which
verdict to trust. There is one arbitrator (`modules/arbitrator.py`); here we only change what it is asked.

**How to iterate:** copy the last *Try* cell, edit `PROMPT`, `QUESTION` or `context`, and run it. Each try runs
LAYA (local, free) on the same 10 clips, with the detectors listed in two orders (A1 first, then the exact reverse). A judge that reads the evidence
gets the clip right and does not flip when only the order changes. Keep every try, so Run All replays them.

Clips: 5 bona fide (including `11249.wav`) and 5 spoof from the fixed arbitration test case
(notebook 05). Thresholds: each artifact's dev
EER; P(bona fide): Platt on dev; `error_rate_pct`: EER on 2021 LA eval.

In [2]:
from functools import partial

from audio_df.helpers.research import arbitration_notebook
from audio_df.pipelines.arbitration import load_cases

cases, det = load_cases(Path(".."))
try_clips = pd.concat([cases[cases.label == "bonafide"].head(5), cases[cases.label == "spoof"].head(5)])

try_request = partial(arbitration_notebook.try_request, try_clips, det)  # (prompt, question, context)

### Try 1: the current reliability request
Each detector's verdict, P(bona fide) and error rate, listed by name.

In [3]:
PROMPT = ("Several spoofing detectors judged the same clip and disagree. Decide which verdict to trust. "
          "Trust detectors with a lower error_rate_pct more.")
QUESTION = {"decision": {"type": "choice", "instructions": "Which verdict should be trusted for this clip?",
                         "criteria": {"bonafide": "The trustworthy evidence says authentic speech.",
                                      "spoof": "The trustworthy evidence says spoofed speech.",
                                      "uncertain": "The evidence conflicts too much to decide."}}}

def context(r, order):
    return {det[m]["name"]: {"verdict": r[f"v_{m}"], "p_bonafide": round(float(r[f"p_{m}"]), 3),
                             "error_rate_pct": det[m]["error_rate_pct"]} for m in order}

try_request(PROMPT, QUESTION, context)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1520.34it/s]

/Users/zuttome/LocalDocs/Learning/College/NUS/MTech-AIS/PS/GC2/SystemCode/.venv/lib/python3.11/site-packages/laya/agent.py:2157: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


right 3/10 | flips 5 | uncertain 0


,clip,truth,A1,A2,A4,judge,P bonafide,P spoof,P uncertain,flips
0,11249,bonafide,bonafide,spoof,bonafide,bonafide,0.49,0.30,0.20,True
1,3190,bonafide,spoof,spoof,bonafide,spoof,0.10,0.81,0.09,False
2,19614,bonafide,spoof,bonafide,spoof,spoof,0.14,0.74,0.12,True
3,25652,bonafide,spoof,spoof,bonafide,spoof,0.09,0.83,0.08,False
4,6867,bonafide,spoof,bonafide,bonafide,spoof,0.10,0.82,0.08,False
5,310,spoof,spoof,spoof,bonafide,spoof,0.09,0.83,0.09,False
6,29329,spoof,bonafide,spoof,bonafide,bonafide,0.47,0.32,0.21,True
7,26539,spoof,spoof,spoof,bonafide,spoof,0.09,0.81,0.09,False
8,19501,spoof,bonafide,spoof,bonafide,bonafide,0.49,0.31,0.20,True
9,22426,spoof,bonafide,spoof,bonafide,bonafide,0.50,0.30,0.20,True


Try 1 ran with three detectors (A1, A2, A4); from Try 2 on, all six are in the request.


### Try 2: the same request, six detectors
Try 1 unchanged; the clips and evidence now come from all six detectors (`pipelines/arbitration.py`).

In [3]:
PROMPT = ("Several spoofing detectors judged the same clip and disagree. Decide which verdict to trust. "
          "Trust detectors with a lower error_rate_pct more.")
QUESTION = {"decision": {"type": "choice", "instructions": "Which verdict should be trusted for this clip?",
                         "criteria": {"bonafide": "The trustworthy evidence says authentic speech.",
                                      "spoof": "The trustworthy evidence says spoofed speech.",
                                      "uncertain": "The evidence conflicts too much to decide."}}}

def context(r, order):
    return {det[m]["name"]: {"verdict": r[f"v_{m}"], "p_bonafide": round(float(r[f"p_{m}"]), 3),
                             "error_rate_pct": det[m]["error_rate_pct"]} for m in order}

try_request(PROMPT, QUESTION, context)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 623.76it/s]

D:\MyCode\projects\GC2-EchoJEV\SystemCode\.venv\Lib\site-packages\laya\agent.py:2157: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


right 5/10 | flips 0 | uncertain 0


,clip,truth,A1,A2,A4,D1,D2,D3,judge,P bonafide,P spoof,P uncertain,flips
0,11249,bonafide,bonafide,spoof,bonafide,spoof,spoof,spoof,spoof,0.19,0.63,0.18,False
1,23944,bonafide,spoof,spoof,bonafide,spoof,spoof,spoof,spoof,0.19,0.64,0.18,False
2,4635,bonafide,bonafide,bonafide,spoof,spoof,spoof,bonafide,spoof,0.25,0.55,0.20,False
3,7387,bonafide,bonafide,spoof,bonafide,spoof,spoof,spoof,spoof,0.21,0.62,0.17,False
4,23400,bonafide,spoof,bonafide,bonafide,spoof,spoof,bonafide,spoof,0.18,0.66,0.16,False
5,15019,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,spoof,0.21,0.61,0.18,False
6,24880,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,spoof,0.22,0.60,0.18,False
7,18295,spoof,bonafide,spoof,spoof,spoof,spoof,spoof,spoof,0.21,0.58,0.21,False
8,22438,spoof,spoof,spoof,bonafide,spoof,spoof,spoof,spoof,0.16,0.71,0.13,False
9,17506,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,spoof,0.21,0.61,0.18,False


### Try 3: detectors grouped by verdict
Try 2 always answers spoof. Here the detectors are grouped under the verdict they gave, sorted by error rate, so the listing order the request receives no longer matters.

In [4]:
PROMPT = ("Several spoofing detectors judged the same clip and disagree. They are grouped by verdict. "
          "Trust the group whose detectors have a lower error_rate_pct.")
QUESTION = {"decision": {"type": "choice", "instructions": "Which verdict should be trusted for this clip?",
                         "criteria": {"bonafide": "The trustworthy evidence says authentic speech.",
                                      "spoof": "The trustworthy evidence says spoofed speech.",
                                      "uncertain": "The evidence conflicts too much to decide."}}}
def context(r, order):
    side = lambda v: {det[m]["name"]: {"p_bonafide": round(float(r[f"p_{m}"]), 3),
                                       "error_rate_pct": det[m]["error_rate_pct"]}
                      for m in sorted(order, key=lambda m: (det[m]["error_rate_pct"], m)) if r[f"v_{m}"] == v}
    return {"say bonafide": side("bonafide"), "say spoof": side("spoof")}
try_request(PROMPT, QUESTION, context)

right 6/10 | flips 0 | uncertain 0


,clip,truth,A1,A2,A4,D1,D2,D3,judge,P bonafide,P spoof,P uncertain,flips
0,11249,bonafide,bonafide,spoof,bonafide,spoof,spoof,spoof,bonafide,0.42,0.32,0.26,False
1,23944,bonafide,spoof,spoof,bonafide,spoof,spoof,spoof,spoof,0.21,0.58,0.20,False
2,4635,bonafide,bonafide,bonafide,spoof,spoof,spoof,bonafide,bonafide,0.45,0.32,0.23,False
3,7387,bonafide,bonafide,spoof,bonafide,spoof,spoof,spoof,bonafide,0.45,0.30,0.24,False
4,23400,bonafide,spoof,bonafide,bonafide,spoof,spoof,bonafide,bonafide,0.42,0.34,0.24,False
5,15019,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,bonafide,0.42,0.34,0.24,False
6,24880,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,bonafide,0.43,0.31,0.27,False
7,18295,spoof,bonafide,spoof,spoof,spoof,spoof,spoof,spoof,0.27,0.48,0.25,False
8,22438,spoof,spoof,spoof,bonafide,spoof,spoof,spoof,spoof,0.22,0.58,0.21,False
9,17506,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,bonafide,0.43,0.35,0.22,False


### Try 4: the three most reliable detectors only
D2, D3 and A2 (2021 LA EER 10-14 %); the LFCC detectors (34-41 %) and D1 are left out. They are listed in a fixed order, and the judge is told to abstain when they disagree.

In [5]:
PROMPT = ("Three reliable spoofing detectors judged the same clip. If they all agree, trust them. "
          "If they disagree, answer uncertain.")
QUESTION = {"decision": {"type": "choice", "instructions": "Which verdict should be trusted for this clip?",
                         "criteria": {"bonafide": "The trustworthy evidence says authentic speech.",
                                      "spoof": "The trustworthy evidence says spoofed speech.",
                                      "uncertain": "The evidence conflicts too much to decide."}}}
def context(r, order):
    return {det[m]["name"]: {"verdict": r[f"v_{m}"], "p_bonafide": round(float(r[f"p_{m}"]), 3)}
            for m in ("d2", "d3", "a2")}
try_request(PROMPT, QUESTION, context)

right 5/10 | flips 0 | uncertain 0


,clip,truth,A1,A2,A4,D1,D2,D3,judge,P bonafide,P spoof,P uncertain,flips
0,11249,bonafide,bonafide,spoof,bonafide,spoof,spoof,spoof,spoof,0.12,0.77,0.11,False
1,23944,bonafide,spoof,spoof,bonafide,spoof,spoof,spoof,spoof,0.11,0.76,0.12,False
2,4635,bonafide,bonafide,bonafide,spoof,spoof,spoof,bonafide,spoof,0.18,0.62,0.20,False
3,7387,bonafide,bonafide,spoof,bonafide,spoof,spoof,spoof,spoof,0.14,0.77,0.09,False
4,23400,bonafide,spoof,bonafide,bonafide,spoof,spoof,bonafide,spoof,0.15,0.67,0.18,False
5,15019,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,spoof,0.11,0.76,0.12,False
6,24880,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,spoof,0.10,0.80,0.10,False
7,18295,spoof,bonafide,spoof,spoof,spoof,spoof,spoof,spoof,0.13,0.77,0.10,False
8,22438,spoof,spoof,spoof,bonafide,spoof,spoof,spoof,spoof,0.12,0.78,0.10,False
9,17506,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,spoof,0.11,0.76,0.12,False


### Try 5: the reliable detectors' vote as a count
The same three detectors, given only as a count, so no detector name or position can lead the answer.

In [6]:
PROMPT = ("Three reliable spoofing detectors judged the same clip. "
          "Follow them when all three agree; answer uncertain when they split.")
QUESTION = {"decision": {"type": "choice", "instructions": "Which verdict should be trusted for this clip?",
                         "criteria": {"bonafide": "The trustworthy evidence says authentic speech.",
                                      "spoof": "The trustworthy evidence says spoofed speech.",
                                      "uncertain": "The evidence conflicts too much to decide."}}}
def context(r, order):
    n = sum(r[f"v_{m}"] == "bonafide" for m in ("d2", "d3", "a2"))
    return {"detectors_saying_bonafide": n, "detectors_saying_spoof": 3 - n}
try_request(PROMPT, QUESTION, context)

right 5/10 | flips 0 | uncertain 0


,clip,truth,A1,A2,A4,D1,D2,D3,judge,P bonafide,P spoof,P uncertain,flips
0,11249,bonafide,bonafide,spoof,bonafide,spoof,spoof,spoof,spoof,0.09,0.79,0.12,False
1,23944,bonafide,spoof,spoof,bonafide,spoof,spoof,spoof,spoof,0.09,0.79,0.12,False
2,4635,bonafide,bonafide,bonafide,spoof,spoof,spoof,bonafide,spoof,0.21,0.65,0.14,False
3,7387,bonafide,bonafide,spoof,bonafide,spoof,spoof,spoof,spoof,0.09,0.79,0.12,False
4,23400,bonafide,spoof,bonafide,bonafide,spoof,spoof,bonafide,spoof,0.21,0.65,0.14,False
5,15019,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,spoof,0.09,0.79,0.12,False
6,24880,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,spoof,0.09,0.79,0.12,False
7,18295,spoof,bonafide,spoof,spoof,spoof,spoof,spoof,spoof,0.09,0.79,0.12,False
8,22438,spoof,spoof,spoof,bonafide,spoof,spoof,spoof,spoof,0.09,0.79,0.12,False
9,17506,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,spoof,0.09,0.79,0.12,False


### Try 3 on all 201 clips
LAYA with Try 3's request on every clip, in both listing orders, against A2 alone and a majority vote of the six detectors (a 3-3 tie counts as uncertain).

In [7]:
scored = arbitration_notebook.try_request(cases, det, PROMPT, QUESTION, context)
votes = sum((cases[f"v_{m}"] == "bonafide").astype(int) for m in det)
answers = {"A2 alone": cases.v_a2,
           "majority vote": np.select([votes > 3, votes < 3], ["bonafide", "spoof"], "uncertain"),
           "LAYA, Try 3": scored.judge.to_numpy()}
pd.DataFrame({k: summarise(v, cases.label) for k, v in answers.items()}).T.round(3)

right 124/200 | flips 0 | uncertain 0


,coverage,accuracy on decided,errors,abstained
A2 alone,1.000,0.735,53.0,0.0
majority vote,0.915,0.667,61.0,17.0
"LAYA, Try 3",1.000,0.620,76.0,0.0


In [8]:
pd.concat({k: pd.crosstab(cases.label.to_numpy(), np.asarray(v), rownames=["truth"], colnames=["answer"])
           for k, v in answers.items()}, axis=1).fillna(0).astype(int)

A2 alone       majority vote                 LAYA, Try 3      
answer   bonafide spoof      bonafide spoof uncertain    bonafide spoof
truth                                                                  
bonafide       57    43            33    53        14          27    73
spoof          10    90             8    89         3           3    97

Try 3 does not beat A2. On the 200 balanced clips LAYA is right on 124 (0.620) against 0.735 for A2 alone; it never flips and never abstains, and it says spoof on 73 of 100 bona fide clips. That follows the detectors: at their 2019 LA dev thresholds every WavLM detector raises false alarms on In-the-Wild bona fide speech (A2 41 %, D2 64 %, D3 59 %, D1 82 %), so on most bona fide clips the evidence itself says spoof.

### The same request with JEV
Paid: one call per clip not already cached in `data/cache/jev/notebook06_arbitration/`. `RUN_JEV = False` only reads the cache.

In [9]:
RUN_JEV = False
replies = [ask_jev(context(r, tuple(det)), PROMPT, QUESTION, Path("../data/cache/jev/notebook06_arbitration"),
                   run=RUN_JEV) for r in cases.to_dict("records")]
print(f"JEV answers cached: {sum(x is not None for x in replies)}/{len(replies)}")
if all(x is not None for x in replies):
    answers["JEV, Try 3"] = np.array([x["decision"]["choice"] for x in replies])
    print(pd.DataFrame({k: summarise(v, cases.label) for k, v in answers.items()}).T.round(3))

JEV answers cached: 0/200


### Verdicts recalibrated on In-the-Wild
At their 2019 LA dev thresholds the WavLM detectors call many In-the-Wild bona fide clips spoof (A2 41 %, D2 64 %, D3 59 % of all 19,963), so the evidence a judge sees leans to spoof. Here each detector's threshold (EER) and P(bona fide) (Platt) are refit on In-the-Wild in two folds by speaker: a clip's fit comes from the other half of the speakers. The baselines a judge must beat: A2 alone, D2 alone, and a random pick among A2, D2 and D3.

In [3]:
recal = arbitration_notebook.recalibrate(cases, Path(".."))
pick = np.random.default_rng(0).choice(["a2", "d2", "d3"], len(cases))
def baselines(c):
    votes = sum((c[f"v_{m}"] == "bonafide").astype(int) for m in det)
    return {"A2 alone": c.v_a2, "D2 alone": c.v_d2,
            "random A2/D2/D3": np.array([c.at[i, f"v_{m}"] for i, m in zip(c.index, pick)]),
            "majority vote": np.select([votes > 3, votes < 3], ["bonafide", "spoof"], "uncertain")}
pd.concat({k: pd.DataFrame({n: summarise(v, c.label) for n, v in baselines(c).items()}).T
           for k, c in {"dev thresholds": cases, "In-the-Wild, 2 folds": recal}.items()}).round(3)

coverage  accuracy on decided  errors  \
dev thresholds       A2 alone            1.000                0.735    53.0   
                     D2 alone            1.000                0.630    74.0   
                     random A2/D2/D3     1.000                0.680    64.0   
                     majority vote       0.915                0.667    61.0   
In-the-Wild, 2 folds A2 alone            1.000                0.775    45.0   
                     D2 alone            1.000                0.755    49.0   
                     random A2/D2/D3     1.000                0.765    47.0   
                     majority vote       0.900                0.739    47.0   

                                      abstained  
dev thresholds       A2 alone               0.0  
                     D2 alone               0.0  
                     random A2/D2/D3        0.0  
                     majority vote         17.0  
In-the-Wild, 2 folds A2 alone               0.0  
                     D2 alone               0.0  
                     random A2/D2/D3        0.0  
                     majority vote         20.0

### Try 6: Try 3 on the recalibrated verdicts
Try 3 unchanged; the 10 clips now carry the recalibrated verdicts and P(bona fide).

In [4]:
try_request = partial(arbitration_notebook.try_request, recal.loc[try_clips.index], det)
PROMPT = ("Several spoofing detectors judged the same clip and disagree. They are grouped by verdict. "
          "Trust the group whose detectors have a lower error_rate_pct.")
QUESTION = {"decision": {"type": "choice", "instructions": "Which verdict should be trusted for this clip?",
                         "criteria": {"bonafide": "The trustworthy evidence says authentic speech.",
                                      "spoof": "The trustworthy evidence says spoofed speech.",
                                      "uncertain": "The evidence conflicts too much to decide."}}}
def context(r, order):
    side = lambda v: {det[m]["name"]: {"p_bonafide": round(float(r[f"p_{m}"]), 3),
                                       "error_rate_pct": det[m]["error_rate_pct"]}
                      for m in sorted(order, key=lambda m: (det[m]["error_rate_pct"], m)) if r[f"v_{m}"] == v}
    return {"say bonafide": side("bonafide"), "say spoof": side("spoof")}
try_request(PROMPT, QUESTION, context)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 624.62it/s]

D:\MyCode\projects\GC2-EchoJEV\SystemCode\.venv\Lib\site-packages\laya\agent.py:2157: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


right 2/10 | flips 0 | uncertain 0


,clip,truth,A1,A2,A4,D1,D2,D3,judge,P bonafide,P spoof,P uncertain,flips
0,11249,bonafide,bonafide,bonafide,bonafide,bonafide,spoof,bonafide,spoof,0.34,0.46,0.21,False
1,23944,bonafide,spoof,spoof,bonafide,spoof,spoof,spoof,spoof,0.22,0.55,0.23,False
2,4635,bonafide,bonafide,bonafide,spoof,bonafide,bonafide,bonafide,spoof,0.23,0.65,0.13,False
3,7387,bonafide,bonafide,bonafide,spoof,spoof,bonafide,bonafide,spoof,0.32,0.43,0.25,False
4,23400,bonafide,spoof,bonafide,spoof,bonafide,bonafide,bonafide,spoof,0.33,0.45,0.21,False
5,15019,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,bonafide,0.49,0.29,0.22,False
6,24880,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,bonafide,0.47,0.29,0.24,False
7,18295,spoof,bonafide,bonafide,spoof,bonafide,bonafide,bonafide,spoof,0.29,0.52,0.18,False
8,22438,spoof,spoof,bonafide,bonafide,bonafide,spoof,bonafide,spoof,0.38,0.45,0.17,False
9,17506,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,bonafide,0.50,0.27,0.23,False


Try 6 on all 200 clips, against the recalibrated baselines.

In [5]:
scored = arbitration_notebook.try_request(recal, det, PROMPT, QUESTION, context)
answers = {**baselines(recal), "LAYA, Try 6": scored.judge.to_numpy()}
pd.DataFrame({k: summarise(v, recal.label) for k, v in answers.items()}).T.round(3)

right 92/200 | flips 0 | uncertain 0


,coverage,accuracy on decided,errors,abstained
A2 alone,1.0,0.775,45.0,0.0
D2 alone,1.0,0.755,49.0,0.0
random A2/D2/D3,1.0,0.765,47.0,0.0
majority vote,0.9,0.739,47.0,20.0
"LAYA, Try 6",1.0,0.460,108.0,0.0


### Try 7: reliability as accuracy, higher is better
Try 6 sides with the group that has the higher error rates (clip 11249: one D2 over five detectors; the spoofs: A1 and A4 over A2, D2 and D3), as if it read "lower" backwards. Here each detector carries `accuracy_pct` = 100 - error rate instead.

In [4]:
try_request = partial(arbitration_notebook.try_request, recal.loc[try_clips.index], det)
PROMPT = ("Several spoofing detectors judged the same clip and disagree. They are grouped by verdict. "
          "Trust the group whose detectors have a higher accuracy_pct.")
QUESTION = {"decision": {"type": "choice", "instructions": "Which verdict should be trusted for this clip?",
                         "criteria": {"bonafide": "The trustworthy evidence says authentic speech.",
                                      "spoof": "The trustworthy evidence says spoofed speech.",
                                      "uncertain": "The evidence conflicts too much to decide."}}}
def context(r, order):
    side = lambda v: {det[m]["name"]: {"p_bonafide": round(float(r[f"p_{m}"]), 3),
                                       "accuracy_pct": round(100 - det[m]["error_rate_pct"], 1)}
                      for m in sorted(order, key=lambda m: (det[m]["error_rate_pct"], m)) if r[f"v_{m}"] == v}
    return {"say bonafide": side("bonafide"), "say spoof": side("spoof")}
try_request(PROMPT, QUESTION, context)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 713.58it/s]

D:\MyCode\projects\GC2-EchoJEV\SystemCode\.venv\Lib\site-packages\laya\agent.py:2157: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


right 1/10 | flips 0 | uncertain 0


,clip,truth,A1,A2,A4,D1,D2,D3,judge,P bonafide,P spoof,P uncertain,flips
0,11249,bonafide,bonafide,bonafide,bonafide,bonafide,spoof,bonafide,spoof,0.36,0.47,0.17,False
1,23944,bonafide,spoof,spoof,bonafide,spoof,spoof,spoof,spoof,0.18,0.65,0.17,False
2,4635,bonafide,bonafide,bonafide,spoof,bonafide,bonafide,bonafide,spoof,0.32,0.51,0.17,False
3,7387,bonafide,bonafide,bonafide,spoof,spoof,bonafide,bonafide,spoof,0.34,0.46,0.20,False
4,23400,bonafide,spoof,bonafide,spoof,bonafide,bonafide,bonafide,spoof,0.40,0.42,0.18,False
5,15019,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,bonafide,0.43,0.40,0.17,False
6,24880,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,bonafide,0.45,0.36,0.19,False
7,18295,spoof,bonafide,bonafide,spoof,bonafide,bonafide,bonafide,spoof,0.35,0.43,0.22,False
8,22438,spoof,spoof,bonafide,bonafide,bonafide,spoof,bonafide,bonafide,0.45,0.40,0.15,False
9,17506,spoof,bonafide,spoof,bonafide,spoof,spoof,spoof,bonafide,0.47,0.34,0.19,False


Try 7 on all 200 clips, against the recalibrated baselines.

In [5]:
scored = arbitration_notebook.try_request(recal, det, PROMPT, QUESTION, context)
answers = {**baselines(recal), "LAYA, Try 7": scored.judge.to_numpy()}
pd.DataFrame({k: summarise(v, recal.label) for k, v in answers.items()}).T.round(3)

right 97/200 | flips 0 | uncertain 0


,coverage,accuracy on decided,errors,abstained
A2 alone,1.0,0.775,45.0,0.0
D2 alone,1.0,0.755,49.0,0.0
random A2/D2/D3,1.0,0.765,47.0,0.0
majority vote,0.9,0.739,47.0,20.0
"LAYA, Try 7",1.0,0.485,103.0,0.0


Recalibrating on In-the-Wild helps the detectors, not LAYA. A2 alone rises from 0.735 to 0.775 and D2 from 0.630 to 0.755; a random pick among A2, D2 and D3 gets 0.765. LAYA gets 0.460 (Try 6) and 0.485 (Try 7), below chance on balanced clips. In both tries it sides with the smaller group on most clips (clip 11249: D2 alone over five detectors), whatever the wording, so it is not weighing the evidence. No LAYA request so far beats the baselines.

### JEV on the arbitration pipeline
`uv run arbitration` with `judge: jev`: the five request variants of `modules/arbitrator.py`, both listing orders, on the same 200 clips at the 2019 LA dev thresholds (2,000 paid calls, saved in the run's `cases.csv`). `agrees with` is how often JEV gives the same answer as each detector.

In [4]:
JEV_RUN = Path("../runs/E-20261029-032651-718483-arbitration")
jev_cases = pd.read_csv(JEV_RUN / "cases.csv")
from audio_df.modules.arbitrator import VARIANTS
table = pd.DataFrame({**{k: summarise(v, cases.label) for k, v in baselines(cases).items()},
                      **{f"JEV, {v}": summarise(jev_cases[f"{v} | A1 first"], cases.label) for v in VARIANTS}}).T
table["flips"] = pd.Series({f"JEV, {v}": int((jev_cases[f"{v} | A1 first"] != jev_cases[f"{v} | A1 last"]).sum())
                            for v in VARIANTS})
agree = pd.DataFrame({f"JEV, {v}": {det[m]["name"]: (jev_cases[f"{v} | A1 first"] == cases[f"v_{m}"]).mean() for m in det}
                      for v in VARIANTS}).T.add_prefix("agrees with ")
table.join(agree).round(3)

,coverage,accuracy on decided,errors,abstained,flips,agrees with A1 LFCC-GMM,agrees with A2 WavLM,agrees with A4 LFCC-MLP,agrees with D1 LFCC+WavLM concat,agrees with D2 LFCC->WavLM attention,agrees with D3 LFCC<->WavLM attention
A2 alone,1.000,0.735,53.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
D2 alone,1.000,0.630,74.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
random A2/D2/D3,1.000,0.680,64.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
majority vote,0.915,0.667,61.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"JEV, verdicts",0.995,0.648,70.0,1.0,11.0,0.605,0.845,0.270,0.875,0.915,0.930
"JEV, probabilities",0.995,0.608,78.0,1.0,10.0,0.595,0.780,0.220,0.935,0.915,0.860
"JEV, reliability",1.000,0.635,73.0,0.0,2.0,0.540,0.780,0.240,0.860,0.995,0.910
"JEV, reliability order",1.000,0.595,81.0,0.0,8.0,0.590,0.770,0.220,0.910,0.925,0.850
"JEV, combined",0.900,0.589,74.0,20.0,2.0,0.560,0.710,0.135,0.855,0.840,0.775


JEV reads the evidence, unlike LAYA, but it does not beat A2 alone (0.735). Its best request (verdicts) gets 0.648; with error rates it follows the most reliable detector, D2 (same answer on 199 of 200 clips), so it scores like D2 alone (0.635 vs 0.630). Like the detectors at their dev thresholds it leans to spoof: on the probabilities request it calls 75 of 100 bona fide clips spoof and 3 of 100 spoofs bona fide. Listing order changes 2-11 answers per request. A judge that follows D2 can only do as well as D2's threshold, so the next step is JEV on the recalibrated evidence (Try 6), where D2 alone gets 0.755.